**Adaptación de dataset KuaiRand Pure a obp**

En el presente notebook se replica el protocolo de evaluación de Saito et al. (2021) en un segundo dominio, videos de corta duración. 

Se utiliza KuaiRand-Pure (Gao et al., 2022) en lugar de 27K, puesto que ambas comparten el mismo log aleatorio (1 186 059 interacciones) y el log de producción adicional de 27K no registra propensities (probabilidades) conocidas, por lo que no permite aplicar estimadores basados en ponderación por importancia.

En particular, se utiliza el log dado en 'log_random_4_22_to_5_08_pure.csv', donde los videos se expusieron al azar. A diferencia de OBD, KuaiRand no publica las probabilidades de exposición. Sin embargo, como la exposición aleatoria no depende del usuario, la probabilidad de cada acción se puede estimar con su frecuencia empírica en el log.


Cabe mencionar, que el paper de KuaiRand no define un protocolo de OPE en específico, por lo que se proponen métodos para adaptar el dataset mencionado para poder realizar una simulación que permita comparar los estimadores baselines utilizados en OBD.

In [48]:
from pathlib import Path
import numpy as np
import pandas as pd

KUAI_PATH = Path("Contenido/Datasets/KuaiRand-Pure/data")

log = pd.read_csv(KUAI_PATH / "log_random_4_22_to_5_08_pure.csv",
                  usecols=["user_id", "video_id", "is_click", "is_rand"])
assert log["is_rand"].eq(1).all() 

print(f"{len(log):,} impresiones | {log['user_id'].nunique():,} usuarios | "
      f"{log['video_id'].nunique():,} videos | CTR = {log['is_click'].mean():.4f}")

1,186,059 impresiones | 27,285 usuarios | 7,583 videos | CTR = 0.1762


Preprocesamos y definimos la acción por escoger del bandit. 

A priori, se podría definir la acción de forma usual en RL4Rec, como el ítem por recomendar, en este caso sería un video.

Veamos como están distribuidos los videos, es decir cuántas veces se muestra un video en los logs. Para ello se utilizará una muestra aleatoria de los logs.

Además se comparará con otras variables posibles de definir como acción, como por ejemplo, la etiqueta (tag) de cada video, sin pérdida de generalidad, se considerará la primera etiqueta de cada video.



In [49]:
videos = pd.read_csv(KUAI_PATH / "video_features_basic_pure.csv", usecols=["video_id", "tag"])

In [50]:
videos["tag1"] = videos["tag"].str.split(",").str[0].fillna("sin tag")

Aplicamos un leftJoin para considerar los videos (por id) junto con su etiqueta. Notar que en la celda anterior se rellenan las etiquetas vacías como tag:"sin tag"

In [51]:
log = log.merge(videos[["video_id", "tag1"]], on="video_id", how="left")
assert log["tag1"].notna().all() 

acciones = np.sort(log["tag1"].unique())
log["accion"] = log["tag1"].map({t: i for i, t in enumerate(acciones)})
print("Número de acciones:", len(acciones))

Número de acciones: 44


In [52]:
muestra = log.sample(n=10_000, random_state=0)

def apariciones(columna):
    conteo = muestra[columna].value_counts().reindex(log[columna].unique(), fill_value=0)
    return {"acciones": len(conteo), "mediana de apariciones": conteo.median(),
            "% con 0 apariciones": 100 * (conteo == 0).mean(),
            "% con menos de 10": 100 * (conteo < 10).mean()}

pd.DataFrame({"video": apariciones("video_id"), "primer tag": apariciones("tag1")}).T.round(1)

,acciones,mediana de apariciones,% con 0 apariciones,% con menos de 10
video,7583.0,1.0,27.6,100.0
primer tag,44.0,89.5,2.3,9.1


Dada la muestra anterior (10 000 impresiones), los videos están distribuidos de forma dispersa, es decir sparse, en comparativa con los tags por cada video. De este modo, dado el gran volumen de videos distintos, y su baja cantidad de apariciones por video, se propone como acción sustituta entonces, la etiqueta de cada video para definir la acción.

Ahora bien, veamos la cantidad de etiquetas por video para analizar el criterio de cuál etiqueta escoger para definir la acción.

In [53]:
videos["n_tags"] = videos["tag"].str.count(",").add(1).fillna(0).astype(int)
impresiones_por_video = log["video_id"].value_counts()

resumen = videos.assign(impresiones=videos["video_id"].map(impresiones_por_video).fillna(0)) \
                .groupby("n_tags").agg(videos=("video_id", "size"), impresiones=("impresiones", "sum"))
resumen["% videos"] = 100 * resumen["videos"] / resumen["videos"].sum()
resumen["% impresiones"] = 100 * resumen["impresiones"] / resumen["impresiones"].sum()
resumen.round(1)

,videos,impresiones,% videos,% impresiones
n_tags,,,,
0,96,15330,1.3,1.3
1,5696,891554,75.1,75.2
2,1759,273947,23.2,23.1
3,32,5228,0.4,0.4


Luego, se concluye que más de un 75% de las impresiones totales (items=videos observados) corresponden a videos que poseen un único tag. Por lo tanto, se considera representativo de los logs en general. Además, por mantener la integridad de los logs, se decide considerar de igual manera los videos con múltiples tags, escogiendo su primer tag como representativo de la acción asociada a aquel.

In [54]:
print(log.head(3))

   user_id  video_id  is_click  is_rand tag1  accion
0        0      5543         0        1   25      17
1        0      1836         0        1   62      36
2        0       721         0        1   15       6


Por cada tag, según la columna is_click, se agregan las columnas impresiones y CTR, en donde CTR representa la tasa de clicks para cada etiqueta del video, mientras que las impresiones justamente las veces que se muestra un video perteneciente según tag.

Finalmente, se considera el promedio de impresiones por tag, se muestra en la columna media impresiones. Mostramos las 10 primeras.

In [55]:
por_accion = log.groupby("tag1")["is_click"].agg(impresiones="size", CTR="mean")
por_accion["media impresiones"] = por_accion["impresiones"] / len(log)
por_accion.sort_values("impresiones", ascending=False).round(4).head(10)

,impresiones,CTR,media impresiones
tag1,,,
39,215470,0.2269,0.1817
3,118385,0.1264,0.0998
20,105862,0.1203,0.0893
12,95647,0.1982,0.0806
9,66519,0.2510,0.0561
6,58938,0.1929,0.0497
7,54130,0.1663,0.0456
2,46071,0.1295,0.0388
1,41068,0.1326,0.0346


**Parte 2**

Se revisan primero las columnas asociadas a los atributos del usuario presentes.
Como en OBD, el contexto son las características del usuario (user_features_pure.csv). 

Veamos la cantidad de videos por valor de cada feature. Esto con el objetivo de dimensionar las cantidad de columnas por crear en one-hot encoding.

In [56]:
usuarios = pd.read_csv(KUAI_PATH / "user_features_pure.csv")
for col in ["is_video_author", "register_days_range", "onehot_feat4"]:
    contadorGroupBy = usuarios[col].value_counts(dropna=False).sort_index().to_string()
    print(f"{col}:\n{contadorGroupBy}\n")

is_video_author:
0     5009
1    22276

register_days_range:
15-30        145
181-365     1778
31-60        381
366-730     3475
61-90        394
730+       20063
8-14           1
91-180      1048

onehot_feat4:
0.0      3039
1.0     10083
2.0      5061
3.0      3184
4.0      2021
5.0      1062
6.0       862
7.0       379
8.0       401
9.0       270
10.0        9
11.0       16
12.0       14
13.0        8
14.0        2
NaN       874



La tabla muestra el número de valores de cada columna, que determina cuántas columnas se generarán posterior a la realización del one-hot encoding mencionado.

In [57]:


niveles = usuarios.drop(columns="user_id").nunique()
print(niveles.sort_values().to_string())


is_lowactive_period         1
onehot_feat17               2
onehot_feat16               2
onehot_feat0                2
onehot_feat13               2
onehot_feat14               2
onehot_feat12               2
onehot_feat15               2
is_video_author             2
is_live_streamer            2
onehot_feat6                3
onehot_feat11               5
onehot_feat10               5
onehot_feat9                7
friend_user_num_range       7
onehot_feat1                7
register_days_range         8
follow_user_num_range       8
user_active_degree          9
fans_user_num_range         9
onehot_feat4               15
onehot_feat5               34
onehot_feat2               50
onehot_feat7              118
onehot_feat8              454
friend_user_num          1391
onehot_feat3             1471
fans_user_num            2210
follow_user_num          2562
register_days            2813


Ahora veamos qué features son más representativas para considerarlas en one-hot encoding. Nuevamente consideramos una muestra de 10 000 sobre el log.

In [58]:
muestra_u = log[["user_id"]].sample(n=10_000, random_state=0).merge(usuarios, on="user_id")
filas = []
for col in niveles[niveles >= 2].sort_values().index:
    conteo = muestra_u[col].astype(str).value_counts().reindex(usuarios[col].astype(str).unique(), fill_value=0)
    filas.append({"columna": col, "niveles": niveles[col], "mediana apariciones": conteo.median(),
                  "% niveles con menos de 10 apariciones": 100 * (conteo < 10).mean()})
pd.DataFrame(filas).set_index("columna").round(1)

,niveles,mediana apariciones,% niveles con menos de 10 apariciones
columna,,,
onehot_feat17,2,237.0,0.0
is_live_streamer,2,5000.0,0.0
is_video_author,2,5000.0,0.0
onehot_feat15,2,256.0,0.0
onehot_feat16,2,391.0,0.0
onehot_feat14,2,1146.0,0.0
onehot_feat13,2,623.0,0.0
onehot_feat12,2,2773.0,0.0
onehot_feat0,2,5000.0,0.0


Luego, si consideramos como umbral mínimo 10 apariciones por nivel  (el min_samples_leaf=10 del modelo $\hat{q}$ en el benchmark de Saito et al., 2021), las features asociadas a las últimas filas mostradas, no cumplirían aquel requisito. Por lo tanto, considerando también la necesidad de evitar una dimensionalidad excesiva, se decide considerar todas aquellas features con igual o menos de 15 niveles, las cuales permiten controlar la cantidad de columnas creadas con One-hot encoding, conservando aquellas features cuyos niveles aparecen con suficiente frecuencia en las impresiones.

In [59]:
categoricas = [c for c in niveles.index if 2 <= niveles[c] <= 15]

contexto_usuario = pd.get_dummies(usuarios[categoricas].astype(str), drop_first=True).astype("float32")
contexto_usuario.index = usuarios["user_id"]
print(f"{len(categoricas)} columnas transformadas a {contexto_usuario.shape[1]} dummies")

20 columnas transformadas a 88 dummies


Ahora, falta definir la política que genera los logs, en OBD era BTS, por otro lado, KuaiRand incluye un log de producción (log_standard), pero no incluye las probabilidades de selección utilizadas, por lo que no puede utilizarse como $\pi_b$ directamente. Es por ello, que se utilizará una política sintética que presente un sesgo similar a BTS en OBD.

In [60]:
OBD_PATH = Path("Contenido/Datasets/open_bandit_dataset/open_bandit_dataset")
ctr_obd = {pol: pd.read_csv(OBD_PATH / pol / "men" / "men.csv", usecols=["click"])["click"].mean()
           for pol in ["bts", "random"]}
razon_bts = ctr_obd["bts"] / ctr_obd["random"]

ctr_random = log["is_click"].mean()  
objetivo = razon_bts * ctr_random
print(f"CTR en OBD: bts {ctr_obd['bts']:.5f} | random {ctr_obd['random']:.5f} | razón {razon_bts:.3f}")
print(f"Valor real de pi_e en KuaiRand: {ctr_random:.4f} | CTR objetivo bajo pi_b: {objetivo:.4f}")

CTR en OBD: bts 0.00674 | random 0.00512 | razón 1.316
Valor real de pi_e en KuaiRand: 0.1762 | CTR objetivo bajo pi_b: 0.2318


Recordemos que la acción del modelo, en este dataset, corresponderá al tag asociado a cada video. Luego el CTR por acción corresponde al CTR por cada tag.

Además, siguiendo la forma de la ecuación (4) de Rafailov et al. (2023):

 $\pi_r$(a) proporcional a  $\pi_{ref}(a) \cdot$ $\exp^{(r(a)/ \beta)}$.

Se construye una política de logging sintética sesgada, 

 $\pi_b$(a) proporcional a  $p0(a) \cdot$ $\exp^{(\alpha \cdot CTR(a))}$.


Aquí la política de referencia es p0 (tagFrequency), la política aleatoria que generó el log de KuaiRand, y la recompensa es el CTR de cada tag. Así se simula el sesgo de BTS frente a Random de OBD.

In [61]:
from scipy.optimize import brentq

ctr_accion = log.groupby("accion")["is_click"].mean().to_numpy()       
tagFrequency = log["accion"].value_counts(normalize=True).sort_index().to_numpy()  

def politica_logging(alpha):
    pesos = tagFrequency * np.exp(alpha * (ctr_accion - ctr_accion.max()))
    return pesos / pesos.sum()

def ctr_bajo(pi):
    return (pi * ctr_accion).sum()




Ahora se requiere calibrar el sesgo (alpha) para emular el sesgo de BTS en OBD, es decir, encontrar el alpha con que el CTR bajo $\pi_b$ sea igual al CTR objetivo (0.2318). Primero veamos cómo cambia el CTR bajo $\pi_b$ para distintos valores de alpha.

In [62]:
from scipy.optimize import brentq

alphas_prueba = [0, 10, 20, 30, 40, 50, 100, 500]
pd.DataFrame({"alpha_sesgo": alphas_prueba,
              "CTR bajo Pi_b": [ctr_bajo(politica_logging(a)) for a in alphas_prueba]}).round(4)

,alpha_sesgo,CTR bajo Pi_b
0,0,0.1762
1,10,0.1952
2,20,0.2111
3,30,0.2227
4,40,0.2311
5,50,0.2379
6,100,0.2723
7,500,0.2935


Ahora calibramos con brentq de scipy el sesgo para emular la política de logging $\pi_b$ que nos da el CTR objetivo bajo $\pi_b$, junto con el $\alpha$ que permite hallar el sesgo requerido para efectuar la simulación comparativa con OBD.

In [63]:
alpha = brentq(lambda a: ctr_bajo(politica_logging(a)) - objetivo, 0, 500)
pi_b = politica_logging(alpha)
print(f"alpha_sesgo = {alpha:.2f} | CTR bajo Pi_b = {ctr_bajo(pi_b):.4f}")

alpha_sesgo = 41.01 | CTR bajo Pi_b = 0.2318




Acorde a la idea de replay de Li et al. (2011), a partir de un log con exposición aleatoria, se obtiene un log equivalente a otra política.

In [64]:
c = (pi_b / tagFrequency).max()                             
acciones_log = log["accion"].to_numpy()
clics_log = log["is_click"].to_numpy()
usuarios_log = log["user_id"].to_numpy()
prob_aceptar = (pi_b / (c * tagFrequency))[acciones_log]     


def log_pi_b(semilla, n):
    rng = np.random.default_rng(semilla)
    aceptadas = np.flatnonzero(rng.random(len(log)) < prob_aceptar)  # muestreo por rechazo (Li et al., 2011)
    idx = rng.choice(aceptadas, size=n, replace=False)
    a = acciones_log[idx]
    return {"n_rounds": n, "n_actions": len(acciones), "action": a, "reward": clics_log[idx],
            "context": contexto_usuario.loc[usuarios_log[idx]].to_numpy(),
            "position": np.zeros(n, dtype=int), "pscore": pi_b[a]}

Estimamos la cantidad de filas para un N = 10 000

In [65]:
N = 10_000
print(f"Filas aceptadas esperadas por pasada: {len(log) / c:,.0f}")


Filas aceptadas esperadas por pasada: 37,025


***Experimento***


Ahora se replica el experimento realizado en OBD (mismos estimadores, modelo $\hat{q}$, n = 10 000 y 200 semillas) en el dataset KuaiRand Pure. Para ello, se estima el CTR de $\pi_e$ = tagFrequency a partir de los logs de $\pi_b$, con valor de referencia 0,1762, correspondiente al CTR observado en el log aleatorio.


En particular, cabe mencionar que la fila "Ingenuo" refiere al promedio de clics del log sin corregir.

In [66]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier
from obp.ope import (OffPolicyEvaluation, RegressionModel, DirectMethod, InverseProbabilityWeighting,
                     SelfNormalizedInverseProbabilityWeighting, DoublyRobust, SelfNormalizedDoublyRobust,
                     SwitchDoublyRobustTuning, DoublyRobustWithShrinkageTuning)

LAMBDAS = [1, 5, 10, 50, 100, 500, 1000, 5000, 10000]
TUNING = dict(lambdas=LAMBDAS, tuning_method="mse", delta=0.05)

def crear_estimadores():
    return [
        InverseProbabilityWeighting(estimator_name="IPW"),
        SelfNormalizedInverseProbabilityWeighting(estimator_name="SNIPW"),
        DirectMethod(estimator_name="DM"),
        DoublyRobust(estimator_name="DR"),
        SelfNormalizedDoublyRobust(estimator_name="SNDR"),
        SwitchDoublyRobustTuning(**TUNING, estimator_name="Switch-DR"),
        DoublyRobustWithShrinkageTuning(**TUNING, estimator_name="DRos"),
    ]

def iteracion_kuairand(semilla, n):
    fb = log_pi_b(semilla, n)
    modelo_q = RegressionModel(
        n_actions=fb["n_actions"], len_list=1, fitting_method="normal",
        base_model=HistGradientBoostingClassifier(learning_rate=0.01, max_iter=100, max_depth=5,
                                                  min_samples_leaf=10, random_state=semilla))
    q_hat = modelo_q.fit_predict(context=fb["context"], action=fb["action"], reward=fb["reward"],
                                 position=fb["position"], pscore=fb["pscore"],
                                 n_folds=2, random_state=semilla)
    pi_e = np.tile(tagFrequency, (n, 1))[:, :, np.newaxis]    
    ope = OffPolicyEvaluation(bandit_feedback=fb, ope_estimators=crear_estimadores())
    estimaciones = ope.estimate_policy_values(action_dist=pi_e, estimated_rewards_by_reg_model=q_hat)
    estimaciones["Ingenuo"] = fb["reward"].mean()  
    return estimaciones

inicio = time.time()
estimaciones = iteracion_kuairand(0, N)

In [67]:
print(estimaciones)

{'IPW': 0.19269846471251587, 'SNIPW': 0.1892140097791367, 'DM': 0.23404401423362464, 'DR': 0.1890056343911373, 'SNDR': 0.18982003737596487, 'Switch-DR': 0.1890056343911373, 'DRos': 0.19189899122477935, 'Ingenuo': 0.2374}


Ahora se prueba una simulación con diversas semillas, de forma análoga al expermiento realizado en OBD.

In [68]:
N_SEMILLAS = 200

inicio = time.time()
filas = []
for semilla in range(N_SEMILLAS):
    for nombre, v in iteracion_kuairand(semilla, N).items():
        filas.append({"semilla": semilla, "estimador": nombre, "estimado": v})
    if (semilla + 1) % 20 == 0:
        print(f"{semilla + 1}/{N_SEMILLAS} semillas | {(time.time() - inicio) / 60:.1f} min")

resultados_kr = pd.DataFrame(filas)
resultados_kr["error2"] = (resultados_kr["estimado"] - ctr_random) ** 2
Path("resultados").mkdir(exist_ok=True)
resultados_kr.to_csv("resultados/kuairand_n10000.csv", index=False)

rmse = resultados_kr.groupby("estimador")["error2"].mean().pow(0.5).sort_values()
(rmse * 1000).round(3)

20/200 semillas | 2.5 min
40/200 semillas | 5.0 min
60/200 semillas | 7.3 min
80/200 semillas | 9.5 min
100/200 semillas | 11.8 min
120/200 semillas | 14.0 min
140/200 semillas | 16.3 min
160/200 semillas | 18.5 min
180/200 semillas | 20.8 min
200/200 semillas | 23.6 min


estimador
SNIPW        11.876
SNDR         11.918
DRos         11.974
IPW          12.297
Switch-DR    12.395
DR           12.407
DM           50.900
Ingenuo      55.863
Name: error2, dtype: float64

Se resumen las 200 semillas con el RMSE, sesgo y desviación estándar, en % del valor real para poder comparar con OBD. Como en la Tabla 6 de Saito et al. (2021), comparamos cada estimador con el mejor mediante el test de Wilcoxon rank-sum sobre los errores al cuadrado.

In [69]:
from scipy.stats import ranksums

res = pd.read_csv("resultados/kuairand_n10000.csv")
g = res.groupby("estimador")["estimado"]
tabla = pd.DataFrame({
    "RMSE rel. (%)": 100 * np.sqrt(res.groupby("estimador")["error2"].mean()) / ctr_random,
    "Sesgo rel. (%)": 100 * (g.mean() - ctr_random) / ctr_random,
    "Desv. estándar rel. (%)": 100 * g.std() / ctr_random,
}).sort_values("RMSE rel. (%)")

# Wilcoxon (equivalente a Tabla 6 (Saito et al., 2021)))
err = res.pivot(index="semilla", columns="estimador", values="error2")
mejor = tabla.index[0]
tabla[f"p-value vs {mejor}"] = [np.nan if e == mejor else ranksums(err[mejor], err[e]).pvalue
                          for e in tabla.index]
tabla.round(3)

,RMSE rel. (%),Sesgo rel. (%),Desv. estándar rel. (%),p-value vs SNIPW
estimador,,,,
SNIPW,6.742,0.052,6.759,NaN
SNDR,6.766,0.105,6.782,0.856
DRos,6.797,2.239,6.434,0.859
IPW,6.981,0.194,6.995,0.342
Switch-DR,7.036,0.015,7.054,0.593
DR,7.043,0.017,7.061,0.593
DM,28.895,28.777,2.616,0.000
Ingenuo,31.712,31.623,2.370,0.000


De lo anterior, se tiene que los estimadores SNIPW, SNDR, DRos, IPW, Switch-DR y DR tienen un error (RMSE relativo) cercano a 7%. Por otro lado, DM tiene un error de 29%, lo cual resulta inverso al resultado obtenido en OBD, donde DM era el estimador con menor error.

Referencias:

Gao, C., Li, S., Zhang, Y., et al. (2022). KuaiRand: An Unbiased Sequential Recommendation Dataset with Randomly Exposed Videos. CIKM 2022. arXiv:2208.08696.

Li, Lihong, et al. “Unbiased Offline Evaluation of Contextual-Bandit-Based News Article Recommendation Algorithms.” Proceedings of the Fourth ACM International Conference on Web Search and Data Mining, edited by , ACM, 2011, pp. 297–306. Wsdm′11. Crossref, https://doi.org/10.1145/1935826.1935878.

Open Bandit Pipeline (obp), versión 0.5.7. Código del paper: https://github.com/st-tech/zr-obp. La configuración del modelo $\hat{q}$ se basa en benchmark/ope/conf/reg_model_hyperparams/default.yaml.

Rafailov, R., Sharma, A., Mitchell, E., Manning, C. D., Ermon, S., & Finn, C. (2023). Direct preference optimization: Your language model is secretly a reward model. Advances in Neural Information Processing Systems, 36.

Saito, Y., Aihara, S., Matsutani, M., & Narita, Y. (2021). Open Bandit Dataset and Pipeline: Towards Realistic and Reproducible Off-Policy Evaluation. NeurIPS Datasets and Benchmarks Track. arXiv:2008.07146.


Virtanen et al. (2020), SciPy, por brentq. Virtanen, P., Gommers, R., Oliphant, T. E., et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. Nature Methods, 17, 261–272.